> **Paths:** set `ILD_MEDGIFT_ROOT`, `ILD_LUNG_MASK_BASE`, `ILD_MODELS_DIR`, and optionally `ILD_EXPORTS_DIR` in the environment (or a local untracked env file). No machine-local defaults are shipped.

# Hierarchical Ablations (binary-primary)

Controlled **retrain** ablations for the hierarchical ILD-XR pipeline.

**Do not run inside NB01.** Main CV + cascade stay in `01_hierarchical_ild.ipynb`.

| Factor | Settings |
|--------|----------|
| Pretrain | Med3D vs ImageNet-inflate (scratch-like) |
| MixUp | on (`MIXUP_ALPHA`) vs off |
| Label smoothing | on vs 0 |
| SE blocks | on vs off |
| Unfreeze | `layer3,layer4` vs heads-only |

Default: **1 patient-disjoint fold** (`ILD_ABLATION_FULL=0`). Metric: **binary F1 / AUC**.

Output: `Results/exports_3d_seg/hierarchical_ablation.json`


## 1 — Setup (reuse NB01 helpers)


In [ ]:
import os, json, gc, random, warnings
from pathlib import Path
from copy import deepcopy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

warnings.filterwarnings('ignore')

# Load local.env — overwrite ILD_* so edits always apply.
def _load_local_env(path, overwrite_ild=True):
    path = Path(path)
    if not path.is_file():
        return False
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, _, value = line.partition('=')
        key, value = key.strip(), value.strip()
        if overwrite_ild and key.startswith('ILD_'):
            os.environ[key] = value
        else:
            os.environ.setdefault(key, value)
    return True

_ENV_CANDIDATES = [
    Path.cwd() / 'local.env',
    Path.cwd() / 'Experimentations' / 'local.env',
    Path('Experimentations/local.env'),
    Path('local.env'),
    
    
]
_loaded_envs = []
for p in _ENV_CANDIDATES:
    if _load_local_env(p):
        _loaded_envs.append(str(Path(p).resolve()))
print('local.env loaded from:', _loaded_envs if _loaded_envs else 'NONE (using defaults)')

# Prefer Experimentations/ as nb05 path
NB01_PATH = Path('Experimentations/01_hierarchical_ild.ipynb')
if not NB01_PATH.is_file():
    NB01_PATH = Path('01_hierarchical_ild.ipynb')
assert NB01_PATH.is_file(), f'Missing {NB01_PATH}'
# Always also load env next to NB01
if _load_local_env(NB01_PATH.parent / 'local.env'):
    p = str((NB01_PATH.parent / 'local.env').resolve())
    if p not in _loaded_envs:
        _loaded_envs.append(p)
        print('also loaded:', p)

_nb05 = json.loads(NB01_PATH.read_text(encoding='utf-8'))

def run_nb05_cell(cell_id, g=None):
    g = globals() if g is None else g
    src = ''.join(next(c for c in _nb05['cells'] if c.get('id') == cell_id)['source'])
    exec(compile(src, f'nb05:{cell_id}', 'exec'), g)
    print(f'OK reused NB01 cell: {cell_id}')

# Paper defaults (not the old 800/8) if env missing
ABLATION_FULL = os.environ.get('ILD_ABLATION_FULL', '1') == '1'
ABLATION_EPOCHS = int(os.environ.get('ILD_ABLATION_EPOCHS', '12'))
ABLATION_PATCHES = int(os.environ.get('ILD_ABLATION_PATCHES', '2400'))
ABLATION_VAL = int(os.environ.get('ILD_ABLATION_VAL', '600'))
ABLATION_SEED = int(os.environ.get('ILD_ABLATION_SEED', '42'))
RUN_ABLATIONS = os.environ.get('ILD_RUN_ABLATIONS', '1') == '1'

print(f'NB01_PATH={NB01_PATH.resolve()}')
print(f'ABLATION_FULL={ABLATION_FULL} EPOCHS={ABLATION_EPOCHS} '
      f'PATCHES={ABLATION_PATCHES} VAL={ABLATION_VAL} RUN={RUN_ABLATIONS}')
print(f'raw env: FULL={os.environ.get("ILD_ABLATION_FULL")} '
      f'EPOCHS={os.environ.get("ILD_ABLATION_EPOCHS")} '
      f'PATCHES={os.environ.get("ILD_ABLATION_PATCHES")} '
      f'VAL={os.environ.get("ILD_ABLATION_VAL")}')


## 2 — Load NB01 setup / data / patches / model


In [ ]:
# Pull shared pipeline definitions from NB01 (same local.env paths/hyperparams)
run_nb05_cell('setup')

# Re-apply local.env AFTER NB01 setup (NB01 uses setdefault and can leave stale keys)
for p in _ENV_CANDIDATES + [NB01_PATH.parent / 'local.env']:
    _load_local_env(p)

# Ablation budget — paper defaults if env missing
ABLATION_FULL = os.environ.get('ILD_ABLATION_FULL', '1') == '1'
ABLATION_EPOCHS = int(os.environ.get('ILD_ABLATION_EPOCHS', '12'))
ABLATION_PATCHES = int(os.environ.get('ILD_ABLATION_PATCHES', '2400'))
ABLATION_VAL = int(os.environ.get('ILD_ABLATION_VAL', '600'))
ABLATION_SEED = int(os.environ.get('ILD_ABLATION_SEED', '42'))
RUN_ABLATIONS = os.environ.get('ILD_RUN_ABLATIONS', '1') == '1'
RUN_CASCADE = False
RUN_TRAIN = False
print(f'Device={device} MEDGIFT_ROOT exists={os.path.isdir(MEDGIFT_ROOT)}')
print(f'Ablation overrides: FULL={ABLATION_FULL} EPOCHS={ABLATION_EPOCHS} '
      f'PATCHES={ABLATION_PATCHES} VAL={ABLATION_VAL}')
print(f'raw env check: FULL={os.environ.get("ILD_ABLATION_FULL")} '
      f'EPOCHS={os.environ.get("ILD_ABLATION_EPOCHS")} '
      f'PATCHES={os.environ.get("ILD_ABLATION_PATCHES")}')

run_nb05_cell('data-loading')
run_nb05_cell('mask-audit')
run_nb05_cell('patch-mining')
run_nb05_cell('model')
print(f'Ready: {len(patient_records)} series | {len(set(r["group"] for r in patient_records))} patients')


## 3 — Factor matrix (1-fold default)


In [ ]:
def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def make_fold_split(seed=ABLATION_SEED):
    groups = list(sorted(set(r['group'] for r in patient_records)))
    rng = np.random.RandomState(seed)
    rng.shuffle(groups)
    group_has_ild = []
    for g in groups:
        recs = [r for r in patient_records if r['group'] == g]
        group_has_ild.append(1 if any(r['has_ild'] for r in recs) else 0)
    n_splits = min(N_FOLDS, len(groups))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = list(skf.split(groups, group_has_ild))
    if ABLATION_FULL:
        return [(i, [groups[j] for j in tr], [groups[j] for j in va]) for i, (tr, va) in enumerate(folds)]
    tr, va = folds[0]
    return [(0, [groups[j] for j in tr], [groups[j] for j in va])]


class PatchDatasetAbl(Dataset):
    def __init__(self, bank, vol_cache, pid_to_rec, augment=True):
        self.bank, self.vol_cache, self.pid_to_rec, self.augment = bank, vol_cache, pid_to_rec, augment

    def __len__(self):
        return len(self.bank)

    def __getitem__(self, idx):
        rec = self.bank[idx]
        packed = self.vol_cache.get(rec.pid, self.pid_to_rec[rec.pid]['path'])
        if packed is None:
            x = torch.zeros(1, *CLS_PATCH_SIZE)
            return x, torch.tensor(0), torch.tensor(0), torch.tensor(0)
        x = torch.from_numpy(extract_patch(packed['ct_norm'], rec.origin, CLS_PATCH_SIZE)).unsqueeze(0).float()
        if self.augment:
            if torch.rand(1).item() > 0.5:
                x = x.flip(-1)
            if torch.rand(1).item() > 0.5:
                x = x.flip(-2)
        return (
            x,
            torch.tensor(rec.binary_label, dtype=torch.long),
            torch.tensor(rec.hier_label, dtype=torch.long),
            torch.tensor(rec.label, dtype=torch.long),
        )


def _selection_score(yt, yp, pr):
    """Prefer MCC; fall back to AUC. Ignore degenerate all-one-class preds."""
    yt = np.asarray(yt)
    yp = np.asarray(yp)
    if len(yt) == 0:
        return -1.0, {'binary_f1': 0.0, 'binary_auc': float('nan'), 'binary_mcc': 0.0, 'degenerate': True}
    f1 = float(f1_score(yt, yp, average='binary', zero_division=0))
    acc = float(accuracy_score(yt, yp))
    mcc = float(matthews_corrcoef(yt, yp)) if len(set(yt.tolist())) > 1 else 0.0
    try:
        auc = float(roc_auc_score(yt, pr[:, 1])) if pr is not None and len(set(yt.tolist())) > 1 else float('nan')
    except Exception:
        auc = float('nan')
    degenerate = len(set(yp.tolist())) < 2
    # Always-ILD on Normal=100/ILD=80 gives F1~0.615; do not crown that as best.
    if degenerate:
        score = -1.0
    else:
        score = mcc if mcc == mcc else (auc if auc == auc else f1)
    return score, {
        'binary_f1': f1,
        'binary_acc': acc,
        'binary_auc': auc,
        'binary_mcc': mcc,
        'degenerate': bool(degenerate),
        'n_pred_pos': int((yp == 1).sum()),
        'n_pred_neg': int((yp == 0).sum()),
    }


def train_eval_binary(tag, train_loader, val_loader, *, use_se=True, mixup_alpha=None,
                      label_smoothing=None, require_pretrain=True, unfreeze=None, epochs=None):
    """NB01-like discriminative binary-primary ablation train."""
    mixup_alpha = MIXUP_ALPHA if mixup_alpha is None else float(mixup_alpha)
    label_smoothing = LABEL_SMOOTHING if label_smoothing is None else float(label_smoothing)
    unfreeze = UNFREEZE_BLOCKS if unfreeze is None else tuple(unfreeze)
    epochs = ABLATION_EPOCHS if epochs is None else int(epochs)

    global REQUIRE_PRETRAIN
    old_req = REQUIRE_PRETRAIN
    REQUIRE_PRETRAIN = bool(require_pretrain)
    try:
        model = HierarchicalEncoder3D(in_ch=1, use_se=use_se).to(device)
        if require_pretrain:
            model._load_pretrained_weights()
        else:
            print(f'[{tag}] skip Med3D pretrain (ImageNet-inflate / random init)')
    finally:
        REQUIRE_PRETRAIN = old_req

    # Binary-only discriminative schedule (same spirit as fixed NB01)
    for prm in model.parameters():
        prm.requires_grad = True
    for head in (model.hier_head, model.path_head):
        for prm in head.parameters():
            prm.requires_grad = False
    if not unfreeze:
        # heads_only: freeze encoder
        for n, prm in model.named_parameters():
            if not n.startswith('binary_head'):
                prm.requires_grad = False
        for prm in model.binary_head.parameters():
            prm.requires_grad = True
        opt = torch.optim.AdamW(model.binary_head.parameters(), lr=LR_HEAD, weight_decay=FINETUNE_WD)
    else:
        set_trainable_blocks(model, unfreeze)
        for prm in model.binary_head.parameters():
            prm.requires_grad = True
        for head in (model.hier_head, model.path_head):
            for prm in head.parameters():
                prm.requires_grad = False
        stem_params = list(model.stem.parameters()) + list(model.layer1.parameters()) + list(model.layer2.parameters())
        l3 = list(model.layer3.parameters()) if 'layer3' in unfreeze else []
        l4 = list(model.layer4.parameters()) if 'layer4' in unfreeze else []
        groups = []
        if stem_params:
            groups.append({'params': stem_params, 'lr': LR_STEM})
        if l3:
            groups.append({'params': l3, 'lr': LR_LAYER3})
        if l4:
            groups.append({'params': l4, 'lr': LR_LAYER4})
        groups.append({'params': list(model.binary_head.parameters()), 'lr': LR_HEAD})
        opt = torch.optim.AdamW(groups, weight_decay=FINETUNE_WD)

    # Binary criterion: optional label smoothing, else class-balanced focal
    if label_smoothing and label_smoothing > 0:
        bin_crit = LabelSmoothCrossEntropy(smoothing=label_smoothing)
    else:
        bin_crit = nn.CrossEntropyLoss()
        try:
            labs = [int(train_loader.dataset.bank[i].binary_label) for i in range(len(train_loader.dataset))]
            bin_counts = np.bincount(labs, minlength=2).astype(float)
            bin_freq = bin_counts / max(bin_counts.sum(), 1)
            bin_alpha = torch.tensor(1.0 / np.maximum(bin_freq, 1e-6), dtype=torch.float32).to(device)
            bin_alpha = bin_alpha / bin_alpha.sum() * 2
            bin_crit = WeightedFocalLoss(alpha=bin_alpha, gamma=2.0)
        except Exception:
            pass

    best_score, best_state = -1e9, None
    for epoch in range(epochs):
        model.train()
        opt.zero_grad(set_to_none=True)
        for step, (x, y_bin, y_hier, y_orig) in enumerate(train_loader):
            x, y_bin = x.to(device), y_bin.to(device)
            if mixup_alpha > 0:
                x, y_a, y_b, lam = mixup_data(x, y_bin, alpha=mixup_alpha)
                feat = model.extract_features(x)
                loss = mixup_criterion(bin_crit, model.binary_head(feat), y_a, y_b, lam)
            else:
                feat = model.extract_features(x)
                loss = bin_crit(model.binary_head(feat), y_bin)
            (loss / GRAD_ACCUM_STEPS).backward()
            if ((step + 1) % GRAD_ACCUM_STEPS == 0) or ((step + 1) == len(train_loader)):
                opt.step()
                opt.zero_grad(set_to_none=True)

        model.eval()
        yt, yp, pr = [], [], []
        with torch.no_grad():
            for x, y_bin, _, _ in val_loader:
                x = x.to(device)
                pb = F.softmax(model.binary_head(model.extract_features(x)), dim=1)
                yt.extend(y_bin.numpy().tolist())
                yp.extend(pb.argmax(1).cpu().numpy().tolist())
                pr.append(pb.cpu().numpy())
        pr = np.concatenate(pr) if pr else np.zeros((0, 2))
        score, mets = _selection_score(yt, yp, pr)
        if score > best_score:
            best_score = score
            best_state = {**mets, 'epoch': epoch + 1, 'select_score': float(score)}
        deg = ' [DEGEN]' if mets['degenerate'] else ''
        print(
            f"  [{tag}] ep {epoch+1}/{epochs} "
            f"binF1={mets['binary_f1']:.4f} AUC={mets['binary_auc']:.4f} "
            f"MCC={mets['binary_mcc']:.4f}{deg}"
        )

    del model
    cuda_cleanup()
    out = best_state or {
        'binary_f1': 0.0, 'binary_auc': float('nan'), 'binary_mcc': 0.0,
        'select_score': -1.0, 'epoch': 0, 'degenerate': True,
    }
    out['tag'] = tag
    return out


from sklearn.metrics import matthews_corrcoef

# Force budget sync immediately before the run (prevents stale kernel values)
for p in list(globals().get('_ENV_CANDIDATES', [])) + [NB01_PATH.parent / 'local.env']:
    try:
        _load_local_env(p)
    except Exception:
        pass
ABLATION_FULL = os.environ.get('ILD_ABLATION_FULL', '1') == '1'
ABLATION_EPOCHS = int(os.environ.get('ILD_ABLATION_EPOCHS', '12'))
ABLATION_PATCHES = int(os.environ.get('ILD_ABLATION_PATCHES', '2400'))
ABLATION_VAL = int(os.environ.get('ILD_ABLATION_VAL', '600'))
ABLATION_SEED = int(os.environ.get('ILD_ABLATION_SEED', '42'))
RUN_ABLATIONS = os.environ.get('ILD_RUN_ABLATIONS', '1') == '1'

if not RUN_ABLATIONS:
    print('SKIP: ILD_RUN_ABLATIONS=0')
    ablation_results = {'skipped': True}
else:
    folds = make_fold_split()
    print(f'Ablation folds: {len(folds)} (FULL={ABLATION_FULL})')
    print(f'Budget: epochs={ABLATION_EPOCHS} patches={ABLATION_PATCHES} val={ABLATION_VAL}')
    if (not ABLATION_FULL) or ABLATION_PATCHES < 2400 or ABLATION_EPOCHS < 12:
        print('WARNING: light budget still active — expected FULL=True patches=2400 epochs=12')
        print(f'  raw env FULL={os.environ.get("ILD_ABLATION_FULL")} '
              f'EPOCHS={os.environ.get("ILD_ABLATION_EPOCHS")} '
              f'PATCHES={os.environ.get("ILD_ABLATION_PATCHES")}')
    print('Selection: max MCC (skip degenerate all-ILD/all-Normal epochs); report F1/AUC/MCC')
    print('Train: discriminative binary-only (NB01-style), MixUp from ep1 when enabled')

    factors = [
        ('full_model', dict(use_se=True, mixup_alpha=MIXUP_ALPHA, label_smoothing=LABEL_SMOOTHING,
                            require_pretrain=True, unfreeze=UNFREEZE_BLOCKS)),
        ('pretrain_imagenet_inflate', dict(use_se=True, mixup_alpha=MIXUP_ALPHA, label_smoothing=LABEL_SMOOTHING,
                                           require_pretrain=False, unfreeze=UNFREEZE_BLOCKS)),
        ('mixup_off', dict(use_se=True, mixup_alpha=0.0, label_smoothing=LABEL_SMOOTHING,
                           require_pretrain=True, unfreeze=UNFREEZE_BLOCKS)),
        ('label_smoothing_off', dict(use_se=True, mixup_alpha=MIXUP_ALPHA, label_smoothing=0.0,
                                     require_pretrain=True, unfreeze=UNFREEZE_BLOCKS)),
        ('se_off', dict(use_se=False, mixup_alpha=MIXUP_ALPHA, label_smoothing=LABEL_SMOOTHING,
                        require_pretrain=True, unfreeze=UNFREEZE_BLOCKS)),
        ('heads_only', dict(use_se=True, mixup_alpha=MIXUP_ALPHA, label_smoothing=LABEL_SMOOTHING,
                            require_pretrain=True, unfreeze=())),
    ]

    results = {name: [] for name, _ in factors}
    for fold_idx, tr_groups, va_groups in folds:
        train_recs = [r for r in patient_records if r['group'] in tr_groups]
        val_recs = [r for r in patient_records if r['group'] in va_groups]
        print(f'\n=== Ablation fold {fold_idx}: train={len(train_recs)} val={len(val_recs)} ===')

        # Shared banks per fold so factors differ only by model recipe
        train_bank, train_vc = build_hierarchical_patch_bank(
            train_recs, MEDGIFT_ROOT, n_patches=ABLATION_PATCHES, seed=ABLATION_SEED + fold_idx)
        val_bank, val_vc = build_hierarchical_patch_bank(
            val_recs, MEDGIFT_ROOT, n_patches=ABLATION_VAL, seed=ABLATION_SEED + 99 + fold_idx,
            min_per_class=max(10, ABLATION_VAL // 6))
        pid_tr = {r['pid']: r for r in train_recs}
        pid_va = {r['pid']: r for r in val_recs}
        train_loader = DataLoader(
            PatchDatasetAbl(train_bank, train_vc, pid_tr, True),
            batch_size=FEAT_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
        val_loader = DataLoader(
            PatchDatasetAbl(val_bank, val_vc, pid_va, False),
            batch_size=FEAT_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

        for name, kwargs in factors:
            print(f'-- factor: {name}')
            try:
                met = train_eval_binary(
                    f'{name}_f{fold_idx}', train_loader, val_loader, **kwargs)
                met['fold'] = fold_idx
                met['n_train_patches'] = len(train_bank)
                met['n_val_patches'] = len(val_bank)
                results[name].append(met)
            except Exception as e:
                print(f'FAIL {name}: {type(e).__name__}: {e}')
                results[name].append({'tag': name, 'fold': fold_idx, 'error': f'{type(e).__name__}: {e}'})

    def _agg(rows, key):
        vals = [r[key] for r in rows if key in r and isinstance(r[key], (int, float)) and r[key] == r[key]]
        if not vals:
            return None
        return {'mean': float(np.mean(vals)), 'std': float(np.std(vals)), 'values': vals}

    ablation_results = {
        'protocol': 'hierarchical_binary_primary_ablation',
        'notebook': '03_hierarchical_ablation.ipynb',
        'ablation_full': ABLATION_FULL,
        'epochs': ABLATION_EPOCHS,
        'patches': ABLATION_PATCHES,
        'val_patches': ABLATION_VAL,
        'n_folds_run': len(folds),
        'metric_primary': 'binary_mcc',
        'metric_secondary': 'binary_f1',
        'selection': 'max_MCC_skip_degenerate',
        'train_recipe': 'discriminative_binary_only',
        'factors': {},
    }
    for name, rows in results.items():
        ablation_results['factors'][name] = {
            'binary_f1': _agg(rows, 'binary_f1'),
            'binary_auc': _agg(rows, 'binary_auc'),
            'binary_mcc': _agg(rows, 'binary_mcc'),
            'runs': rows,
        }

    out_json = os.path.join(EXPORTS_DIR, 'hierarchical_ablation.json')
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(ablation_results, f, indent=2)
    print(f'\nSaved: {out_json}')
    print(json.dumps({k: ablation_results[k] for k in ablation_results if k != 'factors'}, indent=2))

    base = ablation_results['factors'].get('full_model', {}).get('binary_mcc')
    base_m = base['mean'] if base else None
    base_f = ablation_results['factors'].get('full_model', {}).get('binary_f1')
    base_fm = base_f['mean'] if base_f else None
    for name in [n for n, _ in factors]:
        fac = ablation_results['factors'][name]
        mcc = fac['binary_mcc']['mean'] if fac.get('binary_mcc') else None
        f1 = fac['binary_f1']['mean'] if fac.get('binary_f1') else None
        auc = fac['binary_auc']['mean'] if fac.get('binary_auc') else None
        d_mcc = (mcc - base_m) if (mcc is not None and base_m is not None and name != 'full_model') else None
        d_f1 = (f1 - base_fm) if (f1 is not None and base_fm is not None and name != 'full_model') else None
        print(
            f"  {name:28s} F1={f1} AUC={auc} MCC={mcc} "
            f"deltaMCC={d_mcc} deltaF1={d_f1}"
        )


## 4 — Paper table preview


In [ ]:
# LaTeX ablation table (F1 + MCC). Deltas vs full_model.
factors = ablation_results.get('factors') or {}
base_f1 = (factors.get('full_model', {}).get('binary_f1') or {}).get('mean')
base_mcc = (factors.get('full_model', {}).get('binary_mcc') or {}).get('mean')

print(r'\begin{table}[!t]')
print(r'\centering')
print(r'\caption{Ablations on binary Normal vs Any-ILD (patient-disjoint; selection by MCC).}')
print(r'\label{tab:ablation}')
print(r'\begin{tabular}{lcccc}')
print(r'\toprule')
print(r'Factor & Binary F1 & $\Delta$F1 & MCC & $\Delta$MCC \\')
print(r'\midrule')
for name, block in factors.items():
    f1 = (block.get('binary_f1') or {}).get('mean')
    mcc = (block.get('binary_mcc') or {}).get('mean')
    d_f1 = (f1 - base_f1) if isinstance(f1, float) and isinstance(base_f1, float) and name != 'full_model' else (0.0 if name == 'full_model' else None)
    d_mcc = (mcc - base_mcc) if isinstance(mcc, float) and isinstance(base_mcc, float) and name != 'full_model' else (0.0 if name == 'full_model' else None)
    f1_s = f'{f1:.3f}' if isinstance(f1, float) else 'TBD'
    mcc_s = f'{mcc:.3f}' if isinstance(mcc, float) else 'TBD'
    df_s = f'{d_f1:+.3f}' if isinstance(d_f1, float) else 'TBD'
    dm_s = f'{d_mcc:+.3f}' if isinstance(d_mcc, float) else 'TBD'
    print(f'{name.replace("_", " ")} & {f1_s} & {df_s} & {mcc_s} & {dm_s} \\\\')
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')
